# Your First LLM Call

An LLM API call is simple at its core: you send a list of messages, you get one message back. Everything else in this repo is built on top of that one fact.

**By the end of this notebook you'll have:**
- Made a raw chat completion call and seen the response object
- Understood *why* `temperature`, `top_k`, and `top_p` exist, and empirically seen what they do
- Seen `max_tokens` cap a reply
- Met `chat()`, the reusable wrapper in `llm.py` that every later notebook imports

## Setup

We'll start with the raw OpenAI SDK client, not a wrapper, so you can see exactly what's happening under the hood. `llm.py` gets introduced at the end, once you've written its logic by hand.

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1",
)
MODEL = "llama-3.3-70b-versatile"

## A single call

An LLM call is stateless: there's no memory between calls. You send the full conversation every time, and get one assistant reply back.

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "What is an LLM, in one sentence?"}],
)
print(response.choices[0].message.content)

`response` isn't just text — it's a full object (id, model, token usage, etc.). `.choices[0].message.content` is how you pull the reply out of it.

In [ ]:
response

## Why sampling knobs exist at all

At each step, the model doesn't output a word — it outputs a probability distribution over every possible next token. If you always took the single highest-probability token (**greedy decoding**), you'd get an answer, but it would be dry, repetitive, and prone to looping back on itself once it gets stuck in a locally-likely rut.

`temperature`, `top_k`, and `top_p` are three different ways of controlling how much of that distribution the model is allowed to consider before picking one at random. They all shape the *same* underlying distribution, just in different ways.

## Temperature

`temperature` reshapes the distribution itself, before anything is filtered or sampled. Low temperature sharpens it — the already-likely tokens get pushed even higher, so the model becomes focused and repetitive. High temperature flattens it — unlikely tokens get a real chance, so output becomes more varied (and eventually incoherent). At `temperature=0`, the distribution effectively collapses onto the single most likely token — greedy decoding.

Rather than take that on faith, let's prove it: ask for the same thing 3 times at `temperature=0`, then 3 times at `temperature=1.3`, and look at whether the answers repeat or vary.

In [ ]:
def sample(prompt, n=3, **kwargs):
    return [
        client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            **kwargs,
        ).choices[0].message.content
        for _ in range(n)
    ]

tagline_prompt = "Write a one-sentence tagline for a coffee shop."

In [ ]:
print("temperature=0:")
for r in sample(tagline_prompt, temperature=0):
    print("-", r)

In [ ]:
print("temperature=1.3:")
for r in sample(tagline_prompt, temperature=1.3):
    print("-", r)

At `temperature=0` you should see the exact same tagline three times. At `temperature=1.3`, three different ones. That repetition-vs-variation *is* temperature — nothing more mysterious than that.

## top_k

Where temperature reshapes the whole distribution, `top_k` filters it: keep only the `k` most likely tokens, discard everything else, then sample from what's left. `top_k=1` is identical to greedy decoding (`temperature=0`). `top_k=50` always keeps exactly 50 candidates, no matter how the model actually feels.

That fixed count is also its flaw. If the model is highly confident and only 2 tokens make sense, `top_k=50` still keeps 48 nonsense options in the pool. If the model is genuinely torn between 100 plausible continuations, `top_k=50` arbitrarily throws half of them away. The cutoff doesn't adapt to how certain the model is — it's the same hard number every time.

It's also worth knowing that not every provider exposes every knob. Let's try it against Groq directly:

In [ ]:
try:
    client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": tagline_prompt}],
        extra_body={"top_k": 5},
    )
except Exception as e:
    print(e)

Groq's endpoint rejects it outright — `top_k` isn't part of this stack. It's common elsewhere (Anthropic, most local inference servers), so it's worth knowing what it is, but there's nothing to build with it here.

## top_p

`top_p` (nucleus sampling) filters the same way `top_k` does, but by cumulative probability instead of a fixed count: sort tokens by probability, keep adding them until their combined probability crosses `p`, discard the rest, sample from what remains.

That's the actual advantage over `top_k` — the pool size adapts to how confident the model is. When the model is very sure, the nucleus might be 2 tokens; when it's uncertain, it might be 50. `top_p=0.1` only considers a narrow set of very likely tokens; `top_p=1.0` considers all of them.

To isolate its effect, we keep `temperature=1` fixed here and only change `top_p`.

In [ ]:
print("top_p=0.1:")
for r in sample(tagline_prompt, temperature=1, top_p=0.1):
    print("-", r)

In [ ]:
print("top_p=1.0:")
for r in sample(tagline_prompt, temperature=1, top_p=1.0):
    print("-", r)

Same pattern as temperature: a narrow `top_p` collapses to the same answer every time, a wide `top_p` produces variety.

## How they'd combine

In a full sampling pipeline these apply in sequence: `temperature` reshapes the distribution, `top_k` trims it to a fixed count, `top_p` trims what's left by cumulative probability, and the final token is sampled from whatever remains. Groq only exposes `temperature` and `top_p` — which is the norm for OpenAI-compatible endpoints — so in practice you tune one of those two, not all three.

## max_tokens

A hard cap on reply length. The model doesn't know the cap exists ahead of time — it just gets cut off mid-generation once the limit is hit.

In [ ]:
short = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Explain quantum computing."}],
    max_tokens=20,
)
print(short.choices[0].message.content)

## From raw calls to a reusable wrapper

You've now written `client.chat.completions.create(...)` by hand several times. `llm.py` wraps exactly this pattern — same client, same model, same call — so later notebooks don't repeat it.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

from llm import chat

chat([{"role": "user", "content": "What is an LLM, in one sentence?"}])

## What you built

✓ A raw chat completion call, and an understanding of the response object

✓ Proof, by repeated sampling, of what `temperature` and `top_p` actually do — and why `top_k` exists even though this stack doesn't support it

✓ Seen `max_tokens` cut a reply short

✓ Met `chat()` — the reusable wrapper every later notebook will import

**Next:** `02_conversations.ipynb` — turning single calls into multi-turn conversations.